<a href="https://colab.research.google.com/github/MarianoVIsabella/Data-Warehouse-Project/blob/main/DataQuality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Quality Assessment for the Football - FIFA World Cup, 1930 - 2022 dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_PATH = '/content/drive/Shareddrives/2026_unical_dwh/lessons_DQ/L1/'

In [2]:
!pip install ydata-profiling missingno scipy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from datetime import datetime, timedelta
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

In [4]:
class DQAReport:

    """
    # Structured, repeatable Data Quality Assessment Report.
    # Evaluates all ISO 25012 dimensions and produces a scoreboard.
    # Specify that the class checks ISO 25012 quality dimensions and returns a summary report.
    """
    def __init__(self, df: pd.DataFrame, table_name: str, primary_key: str = None):

        self.df         = df.copy()
        self.table_name = table_name
        self.pk         = primary_key
        self.results    = {}   # dimension -> {'score': float, 'issues': int, 'details': str}
        # Initialize a dictionary to store the final results for each quality dimension.
        self.flags      = {}   # dimension -> boolean Series (True = issue)
        # Initialize a dictionary to store row-level issue flags for each dimension.

    def check_completeness(self, required_cols: list = None):
        cols = required_cols or self.df.columns.tolist()
        missing_counts = self.df[cols].isnull().sum()
        total_values   = len(self.df) * len(cols)
        total_missing  = missing_counts.sum()
        score          = 1 - (total_missing / total_values)
        flag_rows      = self.df[cols].isnull().any(axis=1)
        self.flags['completeness'] = flag_rows
        self.results['completeness'] = {
            'score'  : round(score, 4),
            'issues' : int(total_missing),
            'details': f"Missing per column: {missing_counts[missing_counts > 0].to_dict()}"
        }

        return self

    def check_uniqueness(self, key_cols: list = None):
        cols   = key_cols or ([self.pk] if self.pk else self.df.columns.tolist())
        dup    = self.df.duplicated(subset=cols, keep=False)
        score  = 1 - (dup.sum() / len(self.df))
        self.flags['uniqueness'] = dup
        self.results['uniqueness'] = {
            'score'  : round(score, 4),
            'issues' : int(dup.sum()),
            'details': f"Duplicate rows on {cols}: {int(dup.sum())}"
        }
        return self

    def check_validity(self, rules: dict):
        """
        # Document the expected structure of the input parameter.
        rules: {col_name: callable -> bool Series}
        # Each key is a column name and each value is a function returning a boolean Series.
        callable returns True for VALID rows
        # The function must return True for valid values and False for invalid ones.
        """

        all_invalid = pd.Series(False, index=self.df.index)
        rule_details = []

        for col, rule_fn in rules.items():
            if col not in self.df.columns:
                continue
            valid_mask   = rule_fn(self.df[col])
            # Apply the validation function to the column and get a boolean mask of valid rows.
            invalid_mask = ~valid_mask & self.df[col].notna()
            # Mark invalid values while excluding missing values from validity errors.
            all_invalid |= invalid_mask
            # Update the cumulative invalid-row mask using logical OR.
            rule_details.append(f"{col}: {int(invalid_mask.sum())} invalid")
            # Add a summary string reporting the number of invalid values for the current column.

        score = 1 - (all_invalid.sum() / len(self.df))
        self.flags['validity'] = all_invalid

        self.results['validity'] = {
            'score'  : round(score, 4),
            'issues' : int(all_invalid.sum()),
            'details': " | ".join(rule_details)
        }

        return self

    def check_consistency(self, rules: list):

        """
        # Document the expected structure of the input parameter.
        rules: list of callables that return bool Series
        # Each rule is a function that takes the DataFrame and returns a boolean Series.
               True = row is CONSISTENT
        # True means the row satisfies the rule, False means the row is inconsistent.
        """

        all_inconsistent = pd.Series(False, index=self.df.index)
        for rule_fn in rules:
            inconsistent = ~rule_fn(self.df)
            all_inconsistent |= inconsistent
            # Update the cumulative inconsistency mask using logical OR.

        score = 1 - (all_inconsistent.sum() / len(self.df))
        self.flags['consistency'] = all_inconsistent

        self.results['consistency'] = {
            'score'  : round(score, 4),
            'issues' : int(all_inconsistent.sum()),
            'details': f"{int(all_inconsistent.sum())} rows violate at least one consistency rule"
        }

        return self


    def check_timeliness(self, date_col: str, max_age_days: int = 365, allow_future: bool = False):
        if date_col not in self.df.columns:
            return self

        now  = pd.Timestamp.now()
        col  = pd.to_datetime(self.df[date_col], errors='coerce')
        stale  = (now - col).dt.days > max_age_days
        future = col > now if not allow_future else pd.Series(False, index=col.index)
        flag   = stale | future
        score  = 1 - (flag.sum() / len(self.df))
        self.flags['timeliness'] = flag
        self.results['timeliness'] = {
            'score'  : round(score, 4),
            'issues' : int(flag.sum()),
            'details': f"Future dates: {int(future.sum())} | Stale (>{max_age_days}d): {int(stale.sum())}"
        }

        return self

    def scorecard(self) -> pd.DataFrame:
        rows = []
        for dim, res in self.results.items():
            emoji = '🟢' if res['score'] >= 0.95 else ('🟡' if res['score'] >= 0.80 else '🔴')
            rows.append({
                'Table'    : self.table_name,
                'Dimension': dim.capitalize(),
                'Score'    : res['score'],
                'Issues'   : res['issues'],
                'Status'   : emoji,
                'Details'  : res['details']
            })

        return pd.DataFrame(rows)

    def overall_score(self) -> float:
        if not self.results:
            return 0.0 # default return value for robustness

        return round(np.mean([v['score'] for v in self.results.values()]), 4)